# Занятие 6. JOIN и объединение данных из нескольких таблиц

## Цель занятия

На прошлом занятии мы изучили связи таблиц и `FOREIGN KEY`.

Сегодня учимся получать данные сразу из нескольких связанных таблиц.

К концу занятия студент должен понимать:

- зачем нужен `JOIN`;
- чем отличается обычный `SELECT` от `JOIN`;
- как объединять родительскую и дочернюю таблицы;
- что такое `INNER JOIN`;
- что такое `LEFT JOIN`;
- как строить простой отчёт из нескольких таблиц.

---

## Зачем нужен JOIN

Когда данные хранятся в нескольких таблицах, часто нужно собрать их в один отчёт.

Пример:

Таблица `users`:

| id | name |
|---|---|
| 1 | Анна |
| 2 | Иван |

Таблица `orders`:

| id | user_id | title | price |
|---|---|---|---|
| 1 | 1 | Ноутбук | 70000 |
| 2 | 1 | Мышь | 1500 |

Если просто посмотреть `orders`, мы увидим `user_id`, но не имя пользователя.

Чтобы получить имя пользователя и заказ в одной строке, нужен `JOIN`.

---

## Что такое INNER JOIN

`INNER JOIN` показывает только те строки, у которых есть связь в обеих таблицах.

Пример:

```sql
SELECT users.name, orders.title, orders.price
FROM users
INNER JOIN orders ON users.id = orders.user_id;
```

Означает:

- взять пользователей;
- взять заказы;
- соединить их там, где `users.id = orders.user_id`.

---

## Что такое LEFT JOIN

`LEFT JOIN` показывает все строки из левой таблицы, даже если во второй таблице нет связанных записей.

Это полезно, если нужно показать всех пользователей, даже если у некоторых нет заказов.

---

## Где это используется в реальной жизни

- HR: кандидат + его собеседования;
- магазин: клиент + его заказы;
- обучение: студент + его оценки;
- спорт: пользователь + его тренировки;
- питание: пользователь + его приёмы пищи.

---

## Зачем это нужно в дипломе

`JOIN` нужен, когда проект становится реальным.

Примеры:

HR-проект:
- вывести имя кандидата и статус собеседования.

Питание:
- вывести пользователя и его продукты.

Спорт:
- вывести пользователя и его тренировки.

Магазин:
- вывести клиента и его заказы.

Без `JOIN` дипломный проект будет слишком простым.

---

## Связь с GitHub

На этом занятии студент должен сохранить:

- solved-ноутбук;
- TODO-ноутбук;
- SQL-запросы с `JOIN`;
- пример отчёта для своей темы диплома.

Рекомендуемая ветка:

```bash
git checkout main
git pull origin main
git checkout -b feature/lesson-06-join-reports
```

Рекомендуемый commit:

```bash
git add .
git commit -m "feat: add joins and sql reports"
git push -u origin feature/lesson-06-join-reports
```

## Ячейка 1. Подключаем sqlite3 и создаём базу

Создаём базу `join_lesson06.db`.

В ней будут связанные таблицы:
- `users`;
- `orders`.

In [ ]:
import sqlite3

connection = sqlite3.connect("join_lesson06.db")
cursor = connection.cursor()

cursor.execute("PRAGMA foreign_keys = ON")

print("База данных подключена")

## Ячейка 2. Создаём таблицу users

Это родительская таблица.

In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    city TEXT
)
""")

connection.commit()

print("Таблица users создана")

## Ячейка 3. Создаём таблицу orders

Это дочерняя таблица.

Поле `user_id` связывает заказ с пользователем.

In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS orders (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER,
    title TEXT,
    price INTEGER,
    FOREIGN KEY (user_id) REFERENCES users(id)
)
""")

connection.commit()

print("Таблица orders создана")

## Ячейка 4. Очищаем таблицы и добавляем пользователей

Сначала очищаем `orders`, потом `users`, потому что заказы зависят от пользователей.

In [ ]:
cursor.execute("DELETE FROM orders")
cursor.execute("DELETE FROM users")

users = [
    ("Анна", "Казань"),
    ("Иван", "Москва"),
    ("Ольга", "Уфа")
]

cursor.executemany(
    "INSERT INTO users (name, city) VALUES (?, ?)",
    users
)

connection.commit()

print("Пользователи добавлены")

## Ячейка 5. Получаем id пользователей и добавляем заказы

У Анны будет 2 заказа.  
У Ивана будет 1 заказ.  
У Ольги заказов не будет — это понадобится для `LEFT JOIN`.

In [ ]:
cursor.execute("SELECT * FROM users")
users_data = cursor.fetchall()

for user in users_data:
    print(user)

anna_id = users_data[0][0]
ivan_id = users_data[1][0]

orders = [
    (anna_id, "Ноутбук", 70000),
    (anna_id, "Мышь", 1500),
    (ivan_id, "Книга Python", 2500)
]

cursor.executemany(
    "INSERT INTO orders (user_id, title, price) VALUES (?, ?, ?)",
    orders
)

connection.commit()

print("Заказы добавлены")

## Ячейка 6. Обычный SELECT из двух таблиц отдельно

Сначала посмотрим таблицы отдельно.

Проблема: в `orders` мы видим только `user_id`, но не имя пользователя.

In [ ]:
print("USERS:")
cursor.execute("SELECT * FROM users")
users_rows = cursor.fetchall()
for row in users_rows:
    print(row)

print("\nORDERS:")
cursor.execute("SELECT * FROM orders")
orders_rows = cursor.fetchall()
for row in orders_rows:
    print(row)

assert len(users_rows) == 3
assert len(orders_rows) == 3

## Ячейка 7. INNER JOIN

Теперь объединим таблицы.

Получим:
- имя пользователя;
- город;
- название заказа;
- цену.

In [ ]:
cursor.execute("""
SELECT users.name, users.city, orders.title, orders.price
FROM users
INNER JOIN orders ON users.id = orders.user_id
""")

join_rows = cursor.fetchall()

for row in join_rows:
    print(row)

assert len(join_rows) == 3

## Ячейка 8. INNER JOIN с WHERE

Получим только заказы пользователей из Казани.

In [ ]:
cursor.execute("""
SELECT users.name, users.city, orders.title, orders.price
FROM users
INNER JOIN orders ON users.id = orders.user_id
WHERE users.city = ?
""", ("Казань",))

kazan_orders = cursor.fetchall()

for row in kazan_orders:
    print(row)

assert len(kazan_orders) == 2

## Ячейка 9. LEFT JOIN

`LEFT JOIN` показывает всех пользователей.

Если у пользователя нет заказа, в полях заказа будет `None`.

Так мы увидим Ольгу, у которой нет заказов.

In [ ]:
cursor.execute("""
SELECT users.name, users.city, orders.title, orders.price
FROM users
LEFT JOIN orders ON users.id = orders.user_id
""")

left_join_rows = cursor.fetchall()

for row in left_join_rows:
    print(row)

assert len(left_join_rows) == 4

## Ячейка 10. Итоговый отчёт и закрытие соединения

Сделаем небольшой отчёт по заказам.

In [ ]:
cursor.execute("""
SELECT users.name, COUNT(orders.id), SUM(orders.price)
FROM users
LEFT JOIN orders ON users.id = orders.user_id
GROUP BY users.id
""")

report = cursor.fetchall()

print("Отчёт по пользователям:")
for row in report:
    print("Пользователь:", row[0], "| Количество заказов:", row[1], "| Сумма:", row[2])

assert len(report) == 3

connection.close()
print("Соединение закрыто")

# Итог занятия 6

Сегодня вы научились:

- объединять таблицы через `INNER JOIN`;
- использовать `LEFT JOIN`;
- получать данные из нескольких таблиц;
- фильтровать объединённые данные;
- строить простой отчёт.

## Что коммитить в GitHub

- solved-ноутбук;
- TODO-ноутбук;
- SQL-запросы с `JOIN`;
- пример отчёта для своей темы.

```bash
git add .
git commit -m "feat: add joins and sql reports"
git push -u origin feature/lesson-06-join-reports
```